In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
%run ../1_setup/utils

In [0]:
## Enter manually the S3 uri to access data.
dbutils.widgets.text("data_archive", "s3 uri", "Data Archive")
dbutils.widgets.text("data_source", "s3 uri", "Data Source")
# e.g. s3://bucket_name/customers/*.csv
data_archive = dbutils.widgets.get("data_archive")
data_source = dbutils.widgets.get("data_source")
print(data_archive)
print(data_source)

## Bronze

In [0]:
df_bronze = spark.read.csv(data_source, header=True, inferSchema=True)
display(df_bronze.limit(5))

In [0]:
df_bronze = df_bronze.withColumn('ingested_at', F.current_timestamp())\
    .withColumn('_source_file', F.col('_metadata.file_path'))\
    .withColumn('_file_size', F.col('_metadata.file_size'))\
    .withColumn('product_id', F.col('product_id').cast(StringType()))

df_bronze.printSchema()

In [0]:
# merge into historical bronze data
historical_data = DeltaTable.forName(spark, f'{catalog}.{bronze_schema}.fact_orders')
historical_data.alias('historical_data').merge(df_bronze.alias('new_data'), 'historical_data.order_id = new_data.order_id AND historical_data.product_id = new_data.product_id AND historical_data.customer_id = new_data.customer_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# create staging table
df_bronze.write.mode('overwrite')\
    .format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{bronze_schema}.staging_fact_orders')

In [0]:
# archive processed data in S3
files = dbutils.fs.ls(data_source)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{data_archive}/{file_info.name}",
        True
    )

In [0]:
%sql 
-- to describe history
-- DESCRIBE HISTORY fmcg.bronze.fact_orders

In [0]:
%sql
-- to rollback changes
-- RESTORE TABLE fmcg.bronze.fact_orders TO VERSION AS OF 0

## Silver

In [0]:
df_bronze_staging = spark.table(f'{catalog}.{bronze_schema}.staging_fact_orders')
display(df_bronze_staging.limit(10))

In [0]:
df_silver = df_bronze.select(
    'order_id',
    'order_placement_date',
    'customer_id',
    'product_id',
    'order_qty'
)

In [0]:
# 1. Keep only rows where order_qty is present
df_silver = df_bronze_staging.filter(F.col("order_qty").isNotNull())

# 2. Clean customer_id → keep numeric, else set to 999999
df_silver = df_silver.withColumn(
    'customer_id',
    F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id'))
     .otherwise('999999')
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_silver = df_silver.withColumn(
    'order_placement_date',
    F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*', '')
)

# 4. Parse order_placement_date using multiple possible formats
df_silver = df_silver.withColumn(
    'order_placement_date',
    F.coalesce(
        F.try_to_date('order_placement_date', 'yyyy/MM/dd'),
        F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
        F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
        F.try_to_date('order_placement_date', 'MMMM dd, yyyy'),
    )
)

# 5. Drop duplicates
df_silver = df_silver.dropDuplicates(['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])


In [0]:
df_products = spark.table(f'{catalog}.{silver_schema}.dim_products')
df_silver = df_silver.join(df_products, on='product_id', how='inner').select(df_silver['*'], df_products['product_code'])

display(df_silver.limit(5))

In [0]:
# create staging table
df_silver.write.mode('overwrite')\
    .format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{silver_schema}.staging_fact_orders')

In [0]:
# merge into historical silver data
historical_data = DeltaTable.forName(spark, f'{catalog}.{silver_schema}.fact_orders')
historical_data.alias('historical_data').merge(df_silver.alias('new_data'), 'historical_data.order_id = new_data.order_id AND historical_data.product_id = new_data.product_id AND historical_data.customer_id = new_data.customer_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Gold

In [0]:
df_silver_staging = spark.table(f'{catalog}.{silver_schema}.staging_fact_orders')

In [0]:
# Standardizing customer attributes to match Parent Company data model
df_gold = (
    df_silver_staging
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn('date', F.trunc('order_placement_date', 'MM'))
    .withColumnRenamed('customer_id','customer_code')
    .withColumnRenamed('order_qty', 'sold_quantity')
)

display(df_gold.limit(5))

In [0]:
# merge into historical gold data
historical_data = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.child_fact_orders')
historical_data.alias('historical_data').merge(df_silver.alias('new_data'), 'historical_data.order_id = new_data.order_id AND historical_data.product_code = new_data.product_code AND historical_data.customer_code = new_data.customer_code').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Merging Child Company with Parent

- Note: We want data for monthly level but child data is on daily level

In [0]:
df_child = spark.sql(f'SELECT date, product_code, customer_code, sold_quantity FROM {catalog}.{gold_schema}.child_fact_orders')
df_child.show(10)

In [0]:
df_monthly = (
    df_child
    # Group at monthly grain by date + product_code + customer_code
    .groupBy('date', 'product_code', 'customer_code')
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )
)
df_monthly.show(5, truncate=False)

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.fact_orders')
gold_parent_delta.alias('parent_gold').merge(df_monthly.alias('child_gold'), 'parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Clean-up

In [0]:
%sql
DROP TABLE fmcg.bronze.staging_fact_orders;
DROP TABLE fmcg.silver.staging_fact_orders;